# log-samples-eval-callback — worked example 1: Dump N samples every K steps

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `log-samples-eval-callback`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A 'log samples every K steps' callback fires whenever `step % eval_every == 0`. On each fire it generates a fixed number of samples and appends a record to a sink (a plain list standing in for wandb). Step 0 fires because `0 % K == 0`, giving a baseline before training begins.

## Worked solution

We implement the canonical cadence callback against a list sink.

1. **Loop the steps.** `for step in range(n_steps)`.
2. **Cadence test.** `step % eval_every == 0` is True at 0, K, 2K, .... This is the modulo-K cadence; step 0 is included by design so the untrained model gets a baseline dump.
3. **Generate + record.** On a fire we build `n_eval` deterministic sample strings and append `{'step': step, 'samples': [...]}` to the sink. In real code `sink.append(d)` becomes `wandb.log(d, step=step)`.
4. **Count fires.** We return the number of appends so the test can verify cadence.

The demo runs 10 steps with `eval_every=3`, prints the steps that fired (0,3,6,9) and the per-fire sample count.

In [ ]:
def run_with_callback(n_steps, eval_every, n_eval, sink):
    n_fires = 0
    for step in range(n_steps):
        if step % eval_every == 0:
            samples = [f'step={step}-sample={i}' for i in range(n_eval)]
            sink.append({'step': step, 'samples': samples})
            n_fires += 1
    return n_fires

sink = []
fires = run_with_callback(n_steps=10, eval_every=3, n_eval=2, sink=sink)
print('fires:', fires)
print('fired at steps:', [d['step'] for d in sink])
print('samples per fire:', len(sink[0]['samples']))